This notebook takes the various nested directories from the previous notebook and constructs a simple torch dataset. Then we serialize this dataset so that we don't have to perform numerical stacking and disk i/o that will reduce speed.

In [1]:
%load_ext autoreload
%autoreload 2

# Torch Dataset

In [2]:
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm

class DistS1Dataset(Dataset):
    def __init__(self, parquet_dir, transform=None):
        self.parquet_dir = Path(parquet_dir)

        # Load and concatenate all Parquet files
        self.df = self._load_parquet_files()

        # Validate the presence of npz_path column
        if 'npz_path' not in self.df.columns:
            raise ValueError("'npz_path' column is required in the parquet files.")

    def _load_parquet_files(self):
        parquet_files = sorted(self.parquet_dir.glob('*.parquet'))
        if not parquet_files:
            raise FileNotFoundError(f"No parquet files found in {self.parquet_dir}")
        df_list = [pd.read_parquet(pf) for pf in parquet_files]
        df = pd.concat(df_list, ignore_index=True)
        df = df.drop_duplicates().reset_index(drop=True)
        return df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        npz_path = row['npz_path']

        # Load the .npz file
        with np.load(npz_path, allow_pickle=False) as npz:
            sample = {key: npz[key] for key in npz.files}

        return sample

In [3]:
%%time

dist_dataset = DistS1Dataset('npz_paths')

# Dataset

## Visualization

In [5]:
for i, data in enumerate(tqdm(dist_dataset)):
    if i > 32:
        break
data['pre_imgs'].shape, data['post_img'].shape, data['acq_dts_float'].shape

  0%|                                   | 33/297421 [00:22<56:37:46,  1.46it/s]


((14, 2, 256, 256), (2, 256, 256), (15,))

In [6]:
data['acq_dts_float']

array([ 8.22397853,  8.25685524,  8.28973196,  8.3226087 ,  8.35548541,
        8.38836216,  8.4212389 ,  9.21028006,  9.24315677,  9.27603352,
        9.30891023,  9.34178697,  9.37466372,  9.40754043, 10.42671861])